# COMP479 Final Project - Coleton Grossman, Thomas Papka and Oliver Schramm

In [24]:
## Package loading
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
import sqlite3
from pathlib import Path
import glob 

## Project Goals
We are Machine Learning Engineering consultants who have been hired by ToastTab to develop a Denial of Service (DoS) attack classifier. ToastTab is a cloud-based restaurant operation service that allows food service employees to process, prepare, and organize restaurant orders in a seamless fashion. ToastTab offers convenience not only for employees, but also for customers. Instead of calling in to a restaurant to make an order, customers can submit an order through ToastTab that gets paid for and sent to the restaurant immediately. Many restaurants and customers now rely fully on ToastTab's products, so it is of paramount importance that ToastTab is protected from DoS attacks as restaurants would lose the capacity to take online orders and organize orders taken in-house. Naturally, our goal is to prevent such attacks from occuring by building a classification model that can identify DoS attacks and trigger the necessary security measures that can keep ToastTab up and running. Of immediate note regarding our goal for this classification model is that we want to ensure the number of false negatives is as low as possible, so recall stands out as a particularly important evaluation metric.

## Data Sources
### 2.1 Context and Technology Choice
To build this classification model for ToastTab, we are training it on a sample of the 'Intrusion detection evaluation dataset' from the University of New Brunswick's Canadian Institute for Cybersecurity. The full dataset contains a week (Monday-Friday) of generated traffic, but our sample is confined exclusively to the traffic from Wednesday as this date contains a variety of DoS attacks including GoldenEye, Hulk, slowloris, slowhttptest, and Heartbleed. 

## Data Engineering Pipeline - Extract, Transform, Load (ETL)
### 3.1 Context and Technology Choice
To extract our data, we decided to build an extraction function that can handle each of the formats we have (CSV, JSON and Parquet) and combine them into one pandas dataframe for staging. If we were to be using the entire 5-day dataset, a database would have been more appropriate. Considering the smaller scale here, a pandas dataframe will work just fine.

In [21]:
# investigating the columns
ids_0 = pd.read_csv('data/ids_0.csv')
ids_0.columns
# Saving this to a list so I don't need to write out all of the actual names
cols = list(ids_0.columns)
print(cols)

[' Destination Port', ' Flow Duration', ' Total Fwd Packets', ' Total Backward Packets', 'Total Length of Fwd Packets', ' Total Length of Bwd Packets', ' Fwd Packet Length Max', ' Fwd Packet Length Min', ' Fwd Packet Length Mean', ' Fwd Packet Length Std', 'Bwd Packet Length Max', ' Bwd Packet Length Min', ' Bwd Packet Length Mean', ' Bwd Packet Length Std', 'Flow Bytes/s', ' Flow Packets/s', ' Flow IAT Mean', ' Flow IAT Std', ' Flow IAT Max', ' Flow IAT Min', 'Fwd IAT Total', ' Fwd IAT Mean', ' Fwd IAT Std', ' Fwd IAT Max', ' Fwd IAT Min', 'Bwd IAT Total', ' Bwd IAT Mean', ' Bwd IAT Std', ' Bwd IAT Max', ' Bwd IAT Min', 'Fwd PSH Flags', ' Bwd PSH Flags', ' Fwd URG Flags', ' Bwd URG Flags', ' Fwd Header Length', ' Bwd Header Length', 'Fwd Packets/s', ' Bwd Packets/s', ' Min Packet Length', ' Max Packet Length', ' Packet Length Mean', ' Packet Length Std', ' Packet Length Variance', 'FIN Flag Count', ' SYN Flag Count', ' RST Flag Count', ' PSH Flag Count', ' ACK Flag Count', ' URG Flag 

In [ ]:
data_path = 'data/'
def extract(data_path: str | Path) -> pd.DataFrame:
    """This extraction function reads all of the CSV, JSON and Parquet files included
    in our dataset and compiles them into a dataframe
    
    Args: data_folder - the path to the location of our data files

    Returns: a Pandas DataFrame with all data files combined
    """
    # initialize an empty dataframe with the columns from our data
    data = pd.DataFrame(columns=cols)

    # build the extraction loops with glob
    for csvfile in glob.glob(f'{data_path}/*.csv'):
        # establish a temporary df to concatenate on 'data'
        tmp_df = pd.read_csv(csvfile)

        # concatenate
        data = pd.concat([data, tmp_df], ignore_index=True)
    
    